# 02 — Read Iceberg Tables with Polars

Demonstrates reading from the `demo.events` Iceberg table using PyIceberg and Polars.

Covers:
- Full table scan
- Predicate pushdown (filter at Iceberg file-scan layer)
- Column projection
- Time travel (read a past snapshot)
- Overwrite pattern (delete + insert for updates)

**Prerequisites:** Run `01_write_iceberg_polars.ipynb` first.

In [1]:
import os

import polars as pl
from pyiceberg.catalog.rest import RestCatalog
from pyiceberg.expressions import EqualTo

NESSIE_URI       = os.environ["NESSIE_URI"]
S3_ENDPOINT      = os.environ["AWS_S3_ENDPOINT"]
S3_ACCESS_KEY    = os.environ["AWS_ACCESS_KEY_ID"]
S3_SECRET_KEY    = os.environ["AWS_SECRET_ACCESS_KEY"]
WAREHOUSE_BUCKET = os.environ["ICEBERG_WAREHOUSE_BUCKET"]
WAREHOUSE_URI    = f"s3://{WAREHOUSE_BUCKET}/warehouse"

catalog = RestCatalog(
    name="nessie",
    **{
        "uri": NESSIE_URI,
        "warehouse": WAREHOUSE_URI,
        "s3.endpoint": S3_ENDPOINT,
        "s3.access-key-id": S3_ACCESS_KEY,
        "s3.secret-access-key": S3_SECRET_KEY,
        "s3.path-style-access": "true",
        "s3.region": "us-east-1",
    },
)

table = catalog.load_table(("demo", "events"))
print("Table:", table.name())
print("Schema:\n", table.schema())

Table: ('nessie', 'demo', 'events')
Schema:
 table {
  1: event_id: optional string
  2: user_id: optional long
  3: event_type: optional string
  4: amount: optional double
  5: ts: optional timestamp
}


/opt/conda/lib/python3.11/site-packages/pyiceberg/utils/deprecated.py:54: DeprecationWarning: Deprecated in 0.8.0, will be removed in 0.9.0. Table.identifier property is deprecated. Please use Table.name() function instead.
  _deprecation_warning(deprecation_notice(deprecated_in, removed_in, help_message))


## 1. Table metadata

In [2]:
history = table.history()
print(f"Snapshots ({len(history)}):")
for snap in history:
    print(f"  id={snap.snapshot_id}  ts={snap.timestamp_ms}")

Snapshots (1):
  id=2116262770699606240  ts=1778692204198


## 2. Full table scan → Polars DataFrame

In [3]:
df = pl.from_arrow(table.scan().to_arrow())

print(f"Shape: {df.shape}")
df

Shape: (7, 5)


event_id,user_id,event_type,amount,ts
str,i64,str,f64,datetime[μs]
"""e006""",105,"""click""",0.0,2024-01-18 09:00:00
"""e007""",101,"""purchase""",99.0,2024-01-18 10:30:00
"""e001""",101,"""click""",0.0,2024-01-15 10:00:00
"""e002""",102,"""view""",0.0,2024-01-15 11:00:00
"""e003""",103,"""click""",0.0,2024-01-16 09:00:00
"""e004""",101,"""purchase""",49.99,2024-01-16 14:00:00
"""e005""",104,"""view""",0.0,2024-01-17 08:00:00


## 3. Predicate pushdown

The filter is evaluated at the Iceberg file-scan layer — only matching Parquet row groups
are read from S3. Much more efficient than loading everything and filtering in Python.

In [14]:
df_clicks = pl.from_arrow(
    table.scan(
        row_filter=EqualTo("event_type", "click"),
        selected_fields=("event_id", "user_id", "ts"),
    ).to_arrow()
)

print(f"Click events: {len(df_clicks)}")
df_clicks

Click events: 6


event_id,user_id,ts
str,i64,datetime[μs]
"""e006""",105,2024-01-18 09:00:00
"""e001""",101,2024-01-15 10:00:00
"""e003""",103,2024-01-16 09:00:00
"""e001""",101,2024-01-15 10:00:00
"""e003""",103,2024-01-16 09:00:00
"""e006""",105,2024-01-18 09:00:00


## 4. Time travel — read a previous snapshot

Iceberg keeps every snapshot immutable. Pass `snapshot_id` to read any past state of the table.

In [15]:
print(len(history))


1


In [9]:
if len(history) < 2:
    print("Only one snapshot — run notebook 01 twice to create snapshot history.")
else:
    first_snapshot_id = history[0].snapshot_id
    df_v1 = pl.from_arrow(
        table.scan(snapshot_id=first_snapshot_id).to_arrow()
    )
    print(f"Rows in first snapshot: {len(df_v1)}")
    df_v1

Only one snapshot — run notebook 01 twice to create snapshot history.


In [10]:
    first_snapshot_id = history[0].snapshot_id
    df_v1 = pl.from_arrow(
        table.scan(snapshot_id=first_snapshot_id).to_arrow()
    )
    print(f"Rows in first snapshot: {len(df_v1)}")
    df_v1

ValueError: Snapshot not found: 6508384649720162157

## 5. Overwrite pattern (row-level update)

PyIceberg doesn't support in-place row mutations. The standard pattern is:
`overwrite()` with a filter — it atomically deletes matching rows and inserts new ones.

In [6]:
import datetime

updated_row = pl.DataFrame(
    {
        "event_id":   ["e002"],
        "user_id":    [102],
        "event_type": ["view"],
        "amount":     [5.99],
        "ts":         [datetime.datetime(2024, 1, 15, 11, 0, 0)],
    }
)

table.overwrite(
    updated_row.to_arrow(),
    overwrite_filter=EqualTo("event_id", "e002"),
)

print("✔ Overwrite complete. Snapshots now:", len(table.history()))

# Verify
df_after = pl.from_arrow(table.scan(row_filter=EqualTo("event_id", "e002")).to_arrow())
print("Updated row:")
df_after

✔ Overwrite complete. Snapshots now: 1
Updated row:


/opt/conda/lib/python3.11/site-packages/pyiceberg/utils/deprecated.py:54: DeprecationWarning: Deprecated in 0.8.0, will be removed in 0.9.0. Table.identifier property is deprecated. Please use Table.name() function instead.
  _deprecation_warning(deprecation_notice(deprecated_in, removed_in, help_message))
/opt/conda/lib/python3.11/site-packages/pyiceberg/utils/deprecated.py:54: DeprecationWarning: Deprecated in 0.8.0, will be removed in 0.9.0. Support for parsing catalog level identifier in Catalog identifiers is deprecated. Please refer to the table using only its namespace and its table name.
  _deprecation_warning(deprecation_notice(deprecated_in, removed_in, help_message))


event_id,user_id,event_type,amount,ts
str,i64,str,f64,datetime[μs]
"""e002""",102,"""view""",5.99,2024-01-15 11:00:00
